In [ ]:
import random
import uuid
from datetime import datetime, timedelta
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()


TARGET_TABLE = "SYNTH_DISEASE_NOTES"
N_ROWS = 500_000  # increase as needed for perf testing

# -------------------------
# DOMAIN VOCAB
# -------------------------

DISEASES = [
    "type 2 diabetes mellitus",
    "congestive heart failure",
    "chronic obstructive pulmonary disease",
    "non–small cell lung cancer",
    "myocardial infarction",
    "rheumatoid arthritis",
    "Crohn's disease",
    "ulcerative colitis",
    "chronic kidney disease",
    "end‑stage renal disease",
    "hypertensive heart disease",
    "Alzheimer's disease",
    "major depressive disorder",
    "generalized anxiety disorder",
    "acute pancreatitis",
    "hepatocellular carcinoma",
    "multiple sclerosis",
    "systemic lupus erythematosus",
]

SYMPTOMS = [
    "shortness of breath",
    "chest pain",
    "worsening fatigue",
    "unintentional weight loss",
    "night sweats",
    "chronic cough",
    "abdominal pain",
    "bloody stools",
    "joint swelling",
    "cognitive decline",
    "frequent urination",
    "lower extremity edema",
]

MEDS = [
    "metformin",
    "insulin glargine",
    "lisinopril",
    "furosemide",
    "atorvastatin",
    "adalimumab",
    "infliximab",
    "pembrolizumab",
    "sertraline",
    "donepezil",
]

SPECIALTIES = [
    "primary care",
    "cardiology",
    "pulmonology",
    "gastroenterology",
    "oncology",
    "rheumatology",
    "nephrology",
    "psychiatry",
    "neurology",
]

SOURCE_TYPES = ["clinical_note", "telehealth_call", "message_thread", "chart_addendum", "pubmed_abstract"]

TEMPLATES_SINGLE = [
    "The patient with {d1} presents today with {s1}.",
    "History is significant for {d1}, currently managed with {m1}.",
    "He has long‑standing {d1} and reports {s1} over the past {days} days.",
    "She denies chest pain but endorses {s1} in the setting of {d1}.",
    "Assessment: {d1} with recent exacerbation characterized by {s1}.",
]

TEMPLATES_MULTI = [
    "The patient carries diagnoses of {d1} and {d2}, now presenting with {s1} and {s2}.",
    "Complex history of {d1}, {d2}, and prior {d3}, currently on {m1} and {m2}.",
    "Follow‑up for {d1} and {d2}; interval worsening of {s1}.",
    "Long‑standing {d1} with comorbid {d2}; today reports {s1} but no {s2}.",
]

TEMPLATES_ABSTRACT = [
    "Background: {d1} and {d2} are common conditions with significant morbidity. "
    "Methods: We conducted a retrospective cohort study. "
    "Results: Patients with {d1} had higher incidence of {d2}.",
    "Objective: To evaluate outcomes in patients with {d1} treated with {m1}. "
    "Conclusion: {d1} remains associated with increased risk of {d2}.",
]

NAMES = [
    "Mr. Smith", "Ms. Johnson", "Mr. Lee", "Ms. Garcia",
    "Mr. Patel", "Ms. Nguyen", "Mr. Brown", "Ms. Davis",
]

def random_date_within(days_back: int = 365) -> datetime:
    offset = random.randint(0, days_back)
    return datetime.utcnow() - timedelta(days=offset)

def make_row(note_id: int) -> dict:
    d1 = random.choice(DISEASES)
    d2 = random.choice(DISEASES)
    while d2 == d1:
        d2 = random.choice(DISEASES)
    d3 = random.choice(DISEASES)

    s1 = random.choice(SYMPTOMS)
    s2 = random.choice(SYMPTOMS)
    m1 = random.choice(MEDS)
    m2 = random.choice(MEDS)
    spec = random.choice(SPECIALTIES)
    src_type = random.choice(SOURCE_TYPES)
    name = random.choice(NAMES)
    visit_date = random_date_within().strftime("%Y-%m-%d")
    days = random.randint(1, 30)

    # Choose template family
    kind = random.choices(
        ["single", "multi", "abstract"],
        weights=[0.5, 0.3, 0.2],
        k=1,
    )[0]

    if kind == "single":
        tmpl = random.choice(TEMPLATES_SINGLE)
        text = tmpl.format(d1=d1, s1=s1, m1=m1, days=days)
    elif kind == "multi":
        tmpl = random.choice(TEMPLATES_MULTI)
        text = tmpl.format(d1=d1, d2=d2, d3=d3, s1=s1, s2=s2, m1=m1, m2=m2)
    else:  # abstract‑style
        tmpl = random.choice(TEMPLATES_ABSTRACT)
        text = tmpl.format(d1=d1, d2=d2, m1=m1)

    # Add some surrounding clinical context
    prefix = f"{name} is a {spec} patient seen on {visit_date}. "
    suffix_options = [
        " Plan includes close follow‑up and lab monitoring.",
        " Further imaging is recommended to evaluate progression.",
        " Discussed risks, benefits, and alternatives in detail.",
        " Will coordinate care with the primary care physician.",
        "",
    ]
    suffix = random.choice(suffix_options)

    full_text = prefix + text + suffix

    # For quick sanity‑checking later, store which diseases we baked in
    return {
        "NOTE_ID": note_id,
        "NOTE_UID": str(uuid.uuid4()),
        "SOURCE_TYPE": src_type,
        "SPECIALTY": spec,
        "VISIT_DATE": visit_date,
        "TEXT": full_text,
        "TRUE_DISEASES": [d1, d2, d3],  # helpful for evaluation vs NER output
    }

def main():
    # -------------------------
    # Build synthetic dataset
    # -------------------------
    records = [make_row(i) for i in range(1, N_ROWS + 1)]
    df = pd.DataFrame(records)

    # -------------------------
    # Write to Snowflake
    # -------------------------
    

    # Ensure we are in the right DB/SCHEMA
    session.sql(f"USE DATABASE DOCS_DB").collect()
    session.sql(f"USE SCHEMA MAIN").collect()

    # Create Snowpark DF and write
    sp_df = session.create_dataframe(df)

    # Overwrite or create the table
    sp_df.write.mode("overwrite").save_as_table(TARGET_TABLE)

    print(f"Wrote {N_ROWS} rows to DOCS_DB.MAIN.{TARGET_TABLE}")



In [ ]:
main()